# Demo â€” End-to-End Music Context Inference (PDF Sec.10 req)
One audio + text â†’ mel/chroma â†’ segment graph â†’ GNN vs CNN (Task2) â†’ GNN-BERT fusion (Task3) â†’ contrastive retrieval (Task4).
Run top-to-bottom on CPU. Models: `results/gnn_task2_real1000.pt`, `cnn_mel_1000.pt`, `fusion_task3.pt`, `contrastive_task4.pt`.

In [ ]:
import sys
from pathlib import Path
import json, tempfile, os
import numpy as np, torch
ROOT = Path.cwd()
if not (ROOT/'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('ROOT:', ROOT)
print('wavs:', sum(1 for _ in (ROOT/'data/raw/gtzan').rglob('*.wav')))
print('npz:', len(list((ROOT/'data/processed').glob('*.npz'))))
print('graphs:', len(list((ROOT/'data/processed/graphs').glob('*.pt'))))
for f in ['results/metrics_task2_real1000.json','results/metrics_task3.json','results/metrics_task4.json']:
    print(' ', f, (ROOT/f).exists())


## Task 2 â€” audio â†’ graph â†’ genre (GNN vs CNN, real1000 canonical)

In [ ]:
from src.audio_features import process_track
from src.graph_builder import graph_from_npz, graph_coherence_score
from src.gnn_model import GraphSAGEGenre, SimpleCNNMel
from src.task2_data import GENRES_10
wav = sorted((ROOT/'data/raw/gtzan/blues').glob('*.wav'))[0]
print('sample:', wav)
out = process_track(wav, window_sec=5.0, mode='fixed')
print('mel:', out['mel'].shape, 'chroma:', out['chroma'].shape)
with tempfile.NamedTemporaryFile(suffix='.npz', delete=False) as tf:
    np.savez_compressed(tf.name, mel=out['mel'], chroma=out['chroma'], mfcc=out['mfcc'])
    tmp = tf.name
g, meta = graph_from_npz(tmp, tau=0.7)
os.unlink(tmp)
print('graph:', meta['num_nodes'], 'nodes,', meta['num_edges'], 'edges, x:', tuple(g.x.shape))
print('chords:', meta['chords'][:8])
print('Sgraph:', round(graph_coherence_score(g.x.numpy(), g.edge_index.numpy()), 3))
in_dim = g.x.shape[1]
gnn = GraphSAGEGenre(in_dim, 64, 2, 10)
gnn.load_state_dict(torch.load(str(ROOT/'results/gnn_task2_real1000.pt'), map_location='cpu'))
gnn.eval()
cnn = SimpleCNNMel(128, 10)
cnn.load_state_dict(torch.load(str(ROOT/'results/cnn_mel_1000.pt'), map_location='cpu'))
cnn.eval()
with torch.no_grad():
    ei = g.edge_index if g.edge_index.numel() else torch.empty((2,0), dtype=torch.long)
    batch = torch.zeros(g.x.shape[0], dtype=torch.long)
    pg = torch.softmax(gnn(g.x, ei, batch), dim=1).squeeze(0)
    mel = out['mel'][:, out['mel'].shape[1]//2-32:out['mel'].shape[1]//2+32]
    mel_t = torch.from_numpy(mel[None,None].astype('float32'))
    pc = torch.softmax(cnn(mel_t), dim=1).squeeze(0)
print('GNN top3:', sorted(zip(GENRES_10, pg.tolist()), key=lambda x: -x[1])[:3])
print('CNN top3:', sorted(zip(GENRES_10, pc.tolist()), key=lambda x: -x[1])[:3])
print('true genre: blues')

## Task 3 â€” GNN-BERT fusion (cross-attention, macro 0.975)
Ablations from `metrics_task3.json`: bert_only 0.539 / gnn_only 0.374 / concat 0.67 / cross 0.975. Below: load cached cross-attention checkpoint + show 3 case studies.

In [ ]:
m3 = json.load(open(ROOT/'results/metrics_task3.json'))
print('Task3 test:', json.dumps(m3['test'], indent=2))
print('labels:', m3['labels'])
ckpt = torch.load(str(ROOT/'results/fusion_task3.pt'), map_location='cpu')
print('fusion ckpt loaded:', list(ckpt.keys()))
cases = json.load(open(ROOT/'results/task3_cases.json'))
for c in cases:
    print(f"- {c['track']} [{c['genre']}] nodes={c['num_nodes']} edges={c['num_edges']}")
    print('  text:', c['text'])
    print('  true:', c['true'], ' pred:', c['pred'])
try:
    from src.bert_encoder import get_tokenizer
    from transformers import AutoModel
    from src.fusion_model import CrossAttentionFusion
    tok = get_tokenizer('distilbert-base-uncased')
    bert = AutoModel.from_pretrained('distilbert-base-uncased').eval()
    texts = [c['text'] for c in cases]
    enc = tok(texts, truncation=True, padding='max_length', max_length=64, return_tensors='pt')
    with torch.no_grad():
        H = bert(**enc).last_hidden_state
    print('BERT live encode ok:', tuple(H.shape))
except Exception as e:
    print('BERT live skipped (offline ok, cached results above):', str(e)[:200])

## Task 4 â€” contrastive retrieval + zero-shot (R@5 0.367/0.35)

In [ ]:
m4 = json.load(open(ROOT/'results/metrics_task4.json'))
mz = json.load(open(ROOT/'results/metrics_task4_zero.json'))
he = json.load(open(ROOT/'results/human_eval_task4.json'))
print('retrieval:', json.dumps(m4, indent=2))
print(f"zero-shot macro {mz['zero_shot']['macro_f1']:.3f} vs supervised Task1 {mz['supervised_task1']['macro_f1']:.3f}")
print(f"human (SIMULATED, replace with 5 real raters): overall {he['overall_mean']}")
ckpt4 = torch.load(str(ROOT/'results/contrastive_task4.pt'), map_location='cpu')
print('contrastive ckpt:', list(ckpt4.keys()))
ex = json.load(open(ROOT/'results/retrieval_examples_task4.json'))
print(f'showing 3 of {len(ex)} retrieval examples:')
for i in ex[:3]:
    print('Q:', i['query'][:160])
    for k, t in enumerate(i['top3'], 1):
        mark = ' <- TRUE' if t['track']==i['true_track'] else ''
        print(f'  top{k} {t["track"]} s={t["score"]}{mark}: {t["text"][:100]}')
    print()

## Summary
- Task2 real1000 canonical: GNN 0.347/0.336 vs CNN 0.50/0.496 (100/genre = 860 real + 140 aug; pure-real deferred, Sec PROGRESS).
- Task3 cross-attention 0.975 >> concat 0.67 >> bert 0.539 >> gnn 0.374.
- Task4 R@5 0.367/0.35, zero-shot 0.319 beats supervised 0.086; human SIMULATED 3.1/5.
- Reproduce: `python -m src.train_task2_real1000`, `python -m src.train_task3`, `python -m src.train_task4`.